In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import s3fs
import os

In [ ]:
# 1. Chargement
path = "/mnt/c/Users/joelt/Documents/Projets DATA/01_analyse_economie/src/business-risk/assets/Predictions_Risques_Survie_2026.parquet"
df = pd.read_parquet(path)

# 2. Calcul des métriques de poids
mem_usage = df.memory_usage(deep=True) / 1024**2 # En Mo
info_df = pd.DataFrame({
    'Type': df.dtypes,
    'Poids (Mo)': mem_usage.drop('Index'), # On ignore le poids de l'index
    'Taux de Vide (%)': (df.isnull().sum() / len(df)) * 100
})

# 3. Affichage du Top 10 des colonnes les plus lourdes
print("--- TOP 10 DES COLONNES LES PLUS LOURDES ---")
display(info_df.sort_values('Poids (Mo)', ascending=False).head(10))

# 4. Poids total en RAM
print(f"\nTotal RAM : {info_df['Poids (Mo)'].sum():.2f} Mo")
print(f"Poids du fichier sur disque : {os.path.getsize(path) / 1024**2:.2f} Mo")

In [ ]:
df.head()

---

In [ ]:
df['fermeture'].value_counts()

---

In [ ]:
# 1. Chargement
path3 = "/mnt/c/Users/joelt/Documents/Projets DATA/01_analyse_economie/src/business-risk/assets/Reporting_Risques_Survie_2026.parquet"
df3 = pd.read_parquet(path3)

In [ ]:
dataframes = {'DF_Initial': df, 'DF_Nouveau': df3}

# --- COMPARATIV DE STRUCTURE ---
print("📊 ANALYSE DU VOLUME ET DE LA RAM")
audit_data = []
for name, d in dataframes.items():
    audit_data.append({
        'Source': name,
        'Lignes': f"{len(d):,}",
        'Colonnes': len(d.columns),
        'RAM (Mo)': round(d.memory_usage(deep=True).sum() / 1024**2, 2),
        'Siren Uniques': d['SIREN'].nunique() if 'SIREN' in d.columns else 'N/A'
    })

display(pd.DataFrame(audit_data))

# --- RECHERCHE DES DIFFÉRENCES DE COLONNES ---
print("\n🔍 DIFFÉRENCES D'ARCHITECTURE")
all_cols = set(df.columns) | set(df3.columns)
col_audit = []
for col in sorted(all_cols):
    col_audit.append({
        'Nom de la Colonne': col,
        'Dans DF1': '✅' if col in df.columns else '❌',

        'Dans DF3': '✅' if col in df3.columns else '❌'
    })

display(pd.DataFrame(col_audit))

In [ ]:
print("Moyenne Prob_1an DF3:", df3['Prob_1an'].mean())

In [ ]:
df.head()

In [ ]:
df3.head()

In [ ]:
# Filtrer pour ne voir que les entreprises fermées
df_fermees = df3[df3['fermeture'] == 1]

print(f"Nombre d'entreprises fermées détectées : {len(df_fermees):,}")

# Afficher les 10 premières lignes pour inspection
df_fermees.head(10)

In [ ]:
df3.dtypes

In [ ]:
# 1. On garde TOUT le DF3 (1,2M de lignes)
df_master = df3.copy()

# 2. OPTIMISATION RADICALE (La survie de ton appli en dépend)
# On passe de 'object' (très lourd) à 'category' (ultra léger)
cols_cat = [
    'Statut_Expert', 'Code postal de l\'établissement', 'Code commune de l\'établissement', 
    'Code du département de l\'établissement', 'Code de la région de l\'établissement', 
    'code_ape', 'libelle_section_ape', 'SIREN'
]
for col in cols_cat:
    df_master[col] = df_master[col].astype('category')

# On réduit la précision des nombres (float64 -> float32)
cols_float = ['Indice_Risque', 'Prob_1an', 'Prob_2ans', 'Prob_3ans', 'latitude', 'longitude']
for col in cols_float:
    df_master[col] = df_master[col].astype('float32')

# On réduit les entiers
df_master['fermeture'] = df_master['fermeture'].astype('int8')
df_master['age_estime'] = df_master['age_estime'].astype('int16')

# 3. SAUVEGARDE UNIQUE
df_master.to_parquet("master_complet_2026.parquet", index=False, compression='snappy')

print(f"✅ Nouveau poids : {df_master.memory_usage(deep=True).sum() / 1024**2:.2f} Mo")

In [ ]:
df_master['Statut_Expert'].value_counts()

In [ ]:
df['Statut_Expert'].value_counts()